<a href="https://colab.research.google.com/github/SyedaNazish-debug/verifier_model/blob/main/VERIFIER_L_EX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Section

**MOUNTING & CONNECTING THE GOOGLE DRIVE:**

In [3]:
import os
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Path linkage to import the dataset:**

In [1]:
DATA_PATH = "/content/drive/MyDrive/verifier_lense_model/combined_news.csv"

**Confirm the existing data path using pandas library:**

In [5]:
DataFrame = pd.read_csv(DATA_PATH)
print(DATA_PATH)

/content/drive/MyDrive/verifier_lense_model/combined_news.csv


Now let's import the github repo to track the model's development

In [10]:
!git clone https://github.com/SyedaNazish-debug/verifier_model.git
%cd verifier_model
import pandas as pd
data = pd.read_csv(DATA_PATH)

fatal: destination path 'verifier_model' already exists and is not an empty directory.
/content/verifier_model


After importing the github repo let's load the dataset from drive with the path

 Importing the most useful library for performing calculations on data for data preprocessing

The next step in the process is to actually load the correct dataset to set up the workflow inside a Google Colab notebook

The final model development workflow in Colab looks like below:

* Dataset validation
* Text cleaning
* Train-test split
* TF-IDF transformation
* Train baseline model
* Train refined/final model
* Evaluate and compare models
* Save the final model (.pkl)
* Push the changes back to GitHub
* Run the dataset verification



In [12]:
import pandas as pd
print("Dataset shape:", data.shape)
data.head()

print(data.columns)
print(data["label"].value_counts())

Dataset shape: (44058, 5)
Index(['title', 'text', 'subject', 'date', 'label'], dtype='object')
label
0    22848
1    21210
Name: count, dtype: int64




* we have an important finding to do:

* Baseline accuracy: 98.36%
* Refined accuracy: 97.81%

The baseline model was clearly learning dataset-specific source/style indicators such as Reuters, Getty Images, Featured Image, Twitter, etc.

The refined model removes some of that shortcut learning, making it a more defensible final approach.



**Step 1: Prepare the final dataset**

In [13]:
data["combined_text"]=(
    data["title"].fillna("")+" "+
    data["text"].fillna("")
)
x=data["combined_text"]
y=data["label"]

print("x shape:",x.shape)
print("y shape:",y.shape)

print("\n Sample combined text:")
print(x.iloc[0][:500])

x shape: (44058,)
y shape: (44058,)

 Sample combined text:
 Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, 


**We have separate title and text columns:**

In [14]:
print("Before removing duplicate:", len(data))
data = data.drop_duplicates(
    subset="combined_text"
).reset_index(drop=True)

print("After removing duplicates:",len(data))

x=data["combined_text"]
y=data["label"]

print("\n final shape of x:", x.shape)
print("\n final shape of y:",y.shape)

print("\n labelled distribustion of values:",y.value_counts())

Before removing duplicate: 44058
After removing duplicates: 38658

 final shape of x: (38658,)

 final shape of y: (38658,)

 labelled distribustion of values: label
1    21196
0    17462
Name: count, dtype: int64


**Step 2: Final cleaning function**

---


Next Step: Apply the refined
cleaning

In [15]:
import re
def clean_text(text):
  text = str(text)

  text = re.sub(r'https?://\s+|www\.\s+', '' , text)

  text = re.sub(r'\breuters\b',' ',text, flags=re.IGNORECASE)
  text = re.sub(r'\bgetty images\b', ' ', text, flags=re.IGNORECASE)
  text = re.sub(r'\bfeatured image\b', ' ', text, flags=re.IGNORECASE)
  text = re.sub(r'\btwitter\.com\b', ' ', text, flags=re.IGNORECASE)
  text = re.sub(r'\bpic\.twitter\b', ' ', text, flags=re.IGNORECASE)

  text = re.sub(r'\[.*?\]\(.*?\)',' ',text)

  text = re.sub(r'\s+',' ',text)

  return  text.strip()


In [16]:
x_clean =  x.apply(clean_text)

print("Original text:", x.iloc[0][:300])
print("\n cleaned text:",x_clean.iloc[0][:300])


Original text:  Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had j

 cleaned text: Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and the very dishonest fake news media. The former reality show star had just


**Next Substep: Remove Duplicate Texts:**

This is important because duplicate news articles can artificially increase model accuracy if similar copies appear in both the training and testing data.

In [17]:
empty_text = (x_clean.str.strip()=="").sum()
print("Empty texts after cleaning:", empty_text)
duplicate_cleaned = x_clean.duplicated().sum()
print("Duplicate cleaned text:", duplicate_cleaned)

check_data = pd.DataFrame({
    "text": x_clean,
    "label": y
})

conflicting_label = (
    check_data.groupby("text")["label"]
    .nunique()
    .gt(1)
    .sum()
)

print("text appreaing with different label:", conflicting_label)

Empty texts after cleaning: 0
Duplicate cleaned text: 5
text appreaing with different label: 0


In [18]:
check_data =pd.DataFrame({
    "text":x_clean,
    "label": y
})

print("Before final duplicate removal:",len(check_data))

check_data = check_data.drop_duplicates(
    subset="text"
).reset_index(drop=True)

print("After final duplicate removal:",len(check_data))

x_final = check_data["text"]
y_final = check_data["label"]

print("\n Final x shape:", x_final.shape)
print('\n Final y shape:', y_final.shape)

print("\n final label distribution:", y_final.value_counts())

Before final duplicate removal: 38658
After final duplicate removal: 38653

 Final x shape: (38653,)

 Final y shape: (38653,)

 final label distribution: label
1    21194
0    17459
Name: count, dtype: int64


In [19]:
from sklearn.model_selection import train_test_split

x_train , x_test , y_train , y_test = train_test_split(
    x_final,
    y_final,
    test_size = 0.20,
    random_state = 42,
    stratify = y_final
)

print("Training samples:",len(x_train))
print("testing samples:",len(x_test))

print("\n training label distribution:", y_train.value_counts())
print("\n Testing label distribution:",y_test.value_counts())

Training samples: 30922
testing samples: 7731

 training label distribution: label
1    16955
0    13967
Name: count, dtype: int64

 Testing label distribution: label
1    4239
0    3492
Name: count, dtype: int64


**Step 3: Train-test split**

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

final_tfidf = TfidfVectorizer(
    stop_words = "english",
    max_features = 50000,
    ngram_range= (1,2)
)

x_train_tfidf = final_tfidf.fit_transform(x_train)
x_test_tfidf = final_tfidf.transform(x_test)

print("Final TF-IDF training shape:",x_train_tfidf.shape)
print("Final TF_IDF testing shape:", x_test_tfidf.shape)

Final TF-IDF training shape: (30922, 50000)
Final TF_IDF testing shape: (7731, 50000)


In [21]:
from sklearn.linear_model import LogisticRegression

final_model = LogisticRegression(
    max_iter = 1000,
    random_state = 42
)

final_model.fit(
    x_train_tfidf,
    y_train
    )
print("Final model training completed sucessfully.")

Final model training completed sucessfully.


In [22]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

final_predictions = final_model.predict(x_test_tfidf)

print("Final model results")
print("Accuracy:",accuracy_score(y_test,final_predictions))
print("Classification:",classification_report(y_test,final_predictions))
print("Confusion matrix:",confusion_matrix(y_test, final_predictions))


Final model results
Accuracy: 0.9793041003751132
Classification:               precision    recall  f1-score   support

           0       0.98      0.97      0.98      3492
           1       0.98      0.99      0.98      4239

    accuracy                           0.98      7731
   macro avg       0.98      0.98      0.98      7731
weighted avg       0.98      0.98      0.98      7731

Confusion matrix: [[3385  107]
 [  53 4186]]


In [23]:
import joblib
joblib.dump(final_model, "VERIFIER LENSE MODEL.pkl")
joblib.dump(final_tfidf,"TF-IDF_VECTORIZIER.pkl")

print("Model saved sucessfully!")
print("Vectorizer saved sucesfully")

Model saved sucessfully!
Vectorizer saved sucesfully


In [24]:
import os

print(os.listdir())

['README.md', '.git', 'dignostics.py', '.vscode', 'dataset_process.py', 'TF-IDF_VECTORIZIER.pkl', 'verifer_model.py', '__pycache__', 'app.py', 'News _dataset', 'VERIFIER LENSE MODEL.pkl']


In [29]:
import joblib
loaded_model = joblib.load("VERIFIER LENSE MODEL.pkl")
loaded_vectorizer =joblib.load("TF-IDF_VECTORIZIER.pkl")
print("MODEL & vectorizer loaded successfully")

MODEL & vectorizer loaded successfully


In [45]:
def predict_news(news_text):
  cleaned_text = clean_text(news_text)

  text_vector = loaded_vectorizer.transform([cleaned_text])

  prediction = loaded_model.predict(text_vector)[0]

  probability = loaded_model.predict_proba(text_vector)[0]

  if prediction == 0:
    result = "FAKE NEWS"
    confidence = probability[0]

  else:
    result = "REAL NEWS"
    confidence = probability[1]

  return result, confidence

In [47]:
sample_text = x_test.iloc[0]

result, confidence = predict_news(sample_text)

print("Prediction:",result)
print("confidence:",round(confidence * 100 , 2),"%")
print("Actual label:",y_test.iloc[0])

Prediction: REAL NEWS
confidence: 85.14 %
Actual label: 1


In [48]:
print(data
.groupby("label")["subject"]
.value_counts()
.head(10))

label  subject        
0      News                9050
       politics            6430
       US_News              783
       left-news            684
       Government News      515
1      politicsNews       11216
       worldnews           9980
Name: count, dtype: int64


In [50]:
for i in range(5):
    sample_text = x_test.iloc[i]
    result,confidence = predict_news(sample_text)
    actual_label = y_test.iloc[i]
    actual_result = (
    "FAKE NEWS" if  actual_label == 0
    else "REAL NEWS"
)

print(f"\n sample {i + 1}")
print("predicted:",result)
print("confidence:",round(confidence * 100 , 2), "%")
print("Actual value of result:", actual_result)


 sample 5
predicted: FAKE NEWS
confidence: 98.6 %
Actual value of result: FAKE NEWS
